# Pretraining your own `GLACIER` model

This notebook shows how to execute the `GLACIER` pretraining pipeline using our 100k Enamine REAL dataset and corresponding teachers (Minimol, MolFormer) from Hugging Face.

**To use your own data**, replace the `smiles_path` and corresponding teacher paths (`teacher1_minimol_path`, teacher2_molformer_path) variables with your local `.tab` and `.joblib` file paths.

In [ ]:
import torch 
import numpy as np 
import lightning.pytorch as pl
from huggingface_hub import hf_hub_download
from huggingface_hub import snapshot_download
import sys
repo_dir = snapshot_download(repo_id="glacier-hf/Glacier-100k-Mi") 
sys.path.append(repo_dir)

## Retrieve pretraining data

This snippet downloads and locally caches the 100k molecular dataset and its associated teacher model embeddings from Hugging Face. 



In [ ]:
repo_id = "glacier-hf/glacier_pretrain_EnamineREAL_100k" 

# Download SMILES (Cached locally in ~/.cache/huggingface/)
smiles_path = hf_hub_download(
    repo_id=repo_id, 
    filename="enamine_100k.tab", 
    repo_type="dataset"
)

# Download Teacher MiniMol embedding
teacher1_minimol_path = hf_hub_download(
    repo_id=repo_id, 
    filename="teachers/MiniMol/minimol_100k.joblib", 
    repo_type="dataset"
)

# Download Teacher 2 MolFormer embedding
teacher2_molformer_path = hf_hub_download(
    repo_id=repo_id, 
    filename="teachers/MoLFormer-XL-both-10pct/MoLFormer-XL-both-10pct_100k.joblib", 
    repo_type="dataset"
)

## Load pretraining data



In [ ]:
from data.utils import load_data 

# Load 100k SMILES 
df = load_data(smiles_path)
smiles_list = df["Drug"].tolist()

# Load Teacher MiniMol embedding
teacher1_minimol_emb = load_data(teacher1_minimol_path) # Expected Shape: [100000, 512]

# Load Teacher 2 MolFormer embedding
teacher2_molformer_emb = load_data(teacher2_molformer_path) # Expected Shape: [100000, 768]


## Prepare input data

In [ ]:
from data.dataloader import SmilesMoleculeDataset, build_dataloader

# Construct teacher embeddings list 
teacher_emb_list = [teacher1_minimol_emb, teacher2_molformer_emb]

# Construct multimodal dataset
dataset = SmilesMoleculeDataset(smiles=smiles_list, teacher_embeddings=teacher_emb_list, is_train=True)

# Initialize dataloader
dataloader = build_dataloader(dataset, batch_size=1024, num_workers=2, shuffle=True)


## Initialize `GLACIER`

This code creates a `GLACIER` model with default parameters defined in `configuration.py`.

In [ ]:
from glacier_student import Glacier

model = Glacier() 

## Training

In [ ]:
# Create trainer
trainer = pl.Trainer(max_epochs=250,
            gradient_clip_val=1.0, 
            gradient_clip_algorithm="norm", 
            enable_checkpointing=True,
            enable_progress_bar=True,
            accelerator="auto", 
            devices="auto",
            logger=True, 
            deterministic=True
        )


# Train the model 
trainer.fit(model, dataloader)

## Save model weights

In [ ]:
checkpoint_dir = "./Glacier-100k-MiMo"
model.save_pretrained(checkpoint_dir)